In [ ]:

import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, 
                             f1_score, classification_report)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

# Pour que les graphiques s'affichent bien dans le notebook
%matplotlib inline
plt.style.use('seaborn-v0_8')

: 

In [ ]:
# Recharge proprement les données depuis la base
db_path = "oil_wells.db"
conn = sqlite3.connect(db_path)

df_prod = pd.read_sql("SELECT * FROM productions", conn)   # nom corrigé !
df_inc  = pd.read_sql("SELECT * FROM incidents", conn)
df_intv = pd.read_sql("SELECT * FROM interventions", conn)
df_wells = pd.read_sql("SELECT * FROM wells", conn)
df_sites = pd.read_sql("SELECT * FROM sites", conn)

conn.close()

# Conversion des dates
df_prod['timestamp'] = pd.to_datetime(df_prod['timestamp'], errors='coerce')
df_inc['date']      = pd.to_datetime(df_inc['date'], errors='coerce')

print(f"Données chargées : {df_prod.shape[0]} lignes de production")

In [ ]:
# Dataset de base
data = df_prod.copy()
#transforme tes données brutes en données intelligentes que le modèle peut utiliser
# Features temporelles
data['year']       = data['timestamp'].dt.year
data['month']      = data['timestamp'].dt.month
data['day_of_week']= data['timestamp'].dt.dayofweek

# Infos puits (version qui marche à tous les coups)
if 'well_id' in df_wells.columns:
    data = data.merge(df_wells[['well_id', 'depth', 'status']], on='well_id', how='left')
elif 'id' in df_wells.columns:
    data = data.merge(df_wells[['id', 'depth', 'status']], 
                      left_on='well_id', right_on='id', how='left')
    data = data.drop('id', axis=1, errors='ignore')
else:
    print("Colonne d'identifiant puits non trouvée !")

print("Merge avec wells OK")

# Encodage statut
le = LabelEncoder()
data['status'] = le.fit_transform(data['status'].fillna('unknown'))

# Incidents dans les 30 jours précédents
def count_incidents_last_30(row):
    well, ts = row['well_id'], row['timestamp']
    if pd.isna(ts): return 0
    mask = (df_inc['well_id'] == well) & \
           (df_inc['date'] >= ts - pd.Timedelta(days=30)) & \
           (df_inc['date'] < ts)
    return len(df_inc[mask])

data['incidents_last_30d'] = data.apply(count_incidents_last_30, axis=1)

# Target classification : incident dans les 30 jours suivants
def has_incident_next_30(row):
    well, ts = row['well_id'], row['timestamp']
    if pd.isna(ts): return 0
    mask = (df_inc['well_id'] == well) & \
           (df_inc['date'] > ts) & \
           (df_inc['date'] <= ts + pd.Timedelta(days=30))
    return 1 if len(df_inc[mask]) > 0 else 0

data['incident_next_30d'] = data.apply(has_incident_next_30, axis=1)

print("Feature engineering terminé !")
data[['well_id', 'quantity_produced', 'incidents_last_30d', 'incident_next_30d']].head()

In [ ]:
print("Colonnes dans df_wells :")
print(df_wells.columns.tolist())
print("\nAperçu des 3 premières lignes :")
display(df_wells.head(3))

In [ ]:
#Préparer les datasets propres pour l’entraînement des modèles Machine Learning, en séparant la régression (quantité produite) et la classification (incident futur).
features = ['flow_rate', 'pressure', 'temperature', 'depth',
            'incidents_last_30d', 'status', 'year', 'month', 'day_of_week']

target_reg = 'quantity_produced'
target_clf = 'incident_next_30d'

# Datasets propres
df_reg = data.dropna(subset=[target_reg] + features).copy()
df_clf = data.dropna(subset=[target_clf] + features).copy()

print(f"Dataset régression     : {df_reg.shape}")
print(f"Dataset classification : {df_clf.shape}")
print(f"Features utilisées ({len(features)}): {features}")

In [ ]:
#modèle de régression pour prédire la quantité produite
print("Entraînement du modèle de RÉGRESSION (prédiction de quantity_produced)...\n")

X_reg = df_reg[features]
y_reg = df_reg[target_reg]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_r, y_train_r)
y_pred_r = rf_reg.predict(X_test_r)

print("RÉSULTATS RÉGRESSION")
print(f"R²  : {r2_score(y_test_r, y_pred_r):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.2f}")   # ← ligne corrigée
print(f"MAE : {mean_absolute_error(y_test_r, y_pred_r):.2f}\n")

# Importance des features
importances_reg = pd.DataFrame({
    'feature': features,
    'importance': rf_reg.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=importances_reg, x='importance', y='feature', palette='viridis')
plt.title('Facteurs critiques influençant la PRODUCTION')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
#modèle de classification pour prédire si un incident 
# surviendra dans les 30 prochains jours 
# pour un puits, et analyser quelles variables influencent
#  le plus le risque.
print("Entraînement du modèle de CLASSIFICATION (risque d'incident dans 30j)...\n")

X_clf = df_clf[features]
y_clf = df_clf[target_clf]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

rf_clf = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train_c, y_train_c)
y_pred_c = rf_clf.predict(X_test_c)

print("RÉSULTATS CLASSIFICATION")
print(f"Accuracy  : {accuracy_score(y_test_c, y_pred_c):.3f}")
print(f"Precision : {precision_score(y_test_c, y_pred_c):.3f}")
print(f"Recall    : {recall_score(y_test_c, y_pred_c):.3f}")
print(f"F1-score  : {f1_score(y_test_c, y_pred_c):.3f}\n")
print(classification_report(y_test_c, y_pred_c, 
                          target_names=['Pas de risque', 'Risque élevé']))

# Importance des features
importances_clf = pd.DataFrame({
    'feature': features,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=importances_clf, x='importance', y='feature', palette='magma')
plt.title('Facteurs critiques déclenchant un INCIDENT')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Sauvegarde des modèles (prêts à être utilisés dans l'app Flask)
# ===================================================================
joblib.dump(rf_reg, 'model_production_predictor.pkl')
joblib.dump(rf_clf, 'model_incident_risk_predictor.pkl')
joblib.dump(features, 'model_features_list.pkl')  # important pour l'app !

print("Modèles sauvegardés :")
print("→ model_production_predictor.pkl")
print("→ model_incident_risk_predictor.pkl")
print("→ model_features_list.pkl")
print("\nÉtape 3 terminée avec succès !")